<a href="https://colab.research.google.com/github/kumarsirish/agentic-rag-workshop/blob/main/Movie-RAG-ADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation with RAG using Google ADK

An **agentic RAG** movie assistant built with Google's **Agent Development Kit (ADK)**. Instead of fixed steps, we give an agent a tool and let it decide what to do.

**How it works**

- **Retrieval (the tool):** movie plots are turned into vectors by `SentenceTransformer` and stored in an in-memory **ChromaDB** collection. We wrap the search as a **tool** the agent can call.
- **The agent:** Gemini reads the question and chooses: call the search tool, or answer. It can search again if the first results were not enough.
- **Grounding:** the agent's instruction tells it to answer using **only** the movies the tool returned.

```
 Your question
      |
      v
 AGENT (Gemini, built with Google ADK)
      |   decides: "I should search for animated family movies"
      v
 TOOL  search_movies_tool(question)     embeddings + ChromaDB
      |   returns the matching movies
      v
 AGENT writes the final answer from the tool result
```

What makes it *agentic*: the **agent chooses the next step**, not our code.

## 0. Setup

In [ ]:
# Run once. (In Colab this takes a couple of minutes.)
%pip install -q google-adk sentence-transformers chromadb gradio

In [ ]:
import os
import json
import urllib.parse
import urllib.request

import chromadb
from sentence_transformers import SentenceTransformer
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from IPython.display import display, HTML

# Settings
EMBEDDING_MODEL = "all-MiniLM-L6-v2"     # turns text into 384 numbers
LLM_MODEL = "gemini-2.5-flash"           # the agent's brain
TOP_K = 5                                # how many movies the tool returns

In [ ]:
# Keys are read from the Colab secrets vault (the key icon on the left).
# Outside Colab, or if the secret is missing, we ask you to paste the Gemini key instead.
from getpass import getpass


def load_colab_secret(name):
    """Copy a secret from the Colab vault into the environment. Return True if it worked."""
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        return True
    except Exception:
        return False


load_colab_secret("GOOGLE_API_KEY")                # Gemini key: https://aistudio.google.com/apikey
load_colab_secret("HF_TOKEN")                      # optional: speeds up the embedding model download

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"      # use the API key, not Google Cloud

print("Google API key found:", "GOOGLE_API_KEY" in os.environ)
print("HF_TOKEN found      :", "HF_TOKEN" in os.environ)

## 1. Our knowledge base

A small list of 32 movies. Each movie is a Python dictionary.
(Ratings are approximate, and the one-line reviews were written for this demo. They are not real quotes.)

In [ ]:
movies = [
    {"title": "Inception", "year": 2010, "genre": "Sci-Fi, Thriller", "rating": 8.8,
     "cast": "Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page",
     "overview": "A thief who steals secrets by entering people's dreams is hired to plant an idea in a target's mind.",
     "review": "Mind-bending and beautifully shot; you will want to watch it twice."},
    {"title": "Interstellar", "year": 2014, "genre": "Sci-Fi, Drama", "rating": 8.7,
     "cast": "Matthew McConaughey, Anne Hathaway, Jessica Chastain",
     "overview": "Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail.",
     "review": "Epic and emotional, with a score that stays with you."},
    {"title": "The Matrix", "year": 1999, "genre": "Sci-Fi, Action", "rating": 8.7,
     "cast": "Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss",
     "overview": "A hacker discovers that reality is a simulation and joins a rebellion against the machines.",
     "review": "Groundbreaking action and a story that makes you question reality."},
    {"title": "Toy Story", "year": 1995, "genre": "Animation, Family", "rating": 8.3,
     "cast": "Tom Hanks, Tim Allen, Don Rickles",
     "overview": "A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room.",
     "review": "Warm, funny and full of heart for kids and adults alike."},
    {"title": "Finding Nemo", "year": 2003, "genre": "Animation, Family", "rating": 8.2,
     "cast": "Albert Brooks, Ellen DeGeneres, Alexander Gould",
     "overview": "A timid clownfish crosses the ocean to rescue his son.",
     "review": "A colorful, touching adventure about a father's love."},
    {"title": "The Shawshank Redemption", "year": 1994, "genre": "Drama", "rating": 9.3,
     "cast": "Tim Robbins, Morgan Freeman, Bob Gunton",
     "overview": "A banker wrongly jailed for murder finds hope and friendship over decades in prison.",
     "review": "A quiet, powerful story about hope that lifts you up."},
    {"title": "The Godfather", "year": 1972, "genre": "Crime, Drama", "rating": 9.2,
     "cast": "Marlon Brando, Al Pacino, James Caan",
     "overview": "The aging head of a crime family hands control of the empire to his reluctant son.",
     "review": "Slow-burning, masterful storytelling about family and power."},
    {"title": "The Dark Knight", "year": 2008, "genre": "Action, Crime", "rating": 9.0,
     "cast": "Christian Bale, Heath Ledger, Aaron Eckhart",
     "overview": "Batman faces the Joker, a chaotic criminal who pushes Gotham City to its limits.",
     "review": "Dark and intense, with an unforgettable villain."},
    {"title": "Forrest Gump", "year": 1994, "genre": "Drama, Romance", "rating": 8.8,
     "cast": "Tom Hanks, Robin Wright, Gary Sinise",
     "overview": "A kind man with a low IQ unknowingly takes part in major moments of American history.",
     "review": "Gentle, moving and quietly funny; a real feel-good classic."},
    {"title": "Parasite", "year": 2019, "genre": "Thriller, Drama", "rating": 8.5,
     "cast": "Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong",
     "overview": "A poor family schemes to get jobs in a wealthy household, with unexpected consequences.",
     "review": "Sharp, darkly funny and full of surprises."},
    {"title": "Spirited Away", "year": 2001, "genre": "Animation, Fantasy", "rating": 8.6,
     "cast": "Rumi Hiiragi, Miyu Irino, Mari Natsuki",
     "overview": "A girl wanders into a world of spirits and must work in a bathhouse to free her parents.",
     "review": "Imaginative and magical; every frame feels hand-crafted."},
    {"title": "The Lion King", "year": 1994, "genre": "Animation, Family", "rating": 8.5,
     "cast": "Matthew Broderick, Jeremy Irons, James Earl Jones",
     "overview": "A young lion prince flees after his father's death and later returns to reclaim his kingdom.",
     "review": "A timeless, emotional story with wonderful songs."},
    {"title": "Mad Max: Fury Road", "year": 2015, "genre": "Action, Sci-Fi", "rating": 8.1,
     "cast": "Tom Hardy, Charlize Theron, Nicholas Hoult",
     "overview": "In a desert wasteland, a drifter and a rebel driver flee a tyrant in one long chase.",
     "review": "Non-stop, breathtaking action with very little dialogue."},
    {"title": "La La Land", "year": 2016, "genre": "Romance, Musical", "rating": 8.0,
     "cast": "Ryan Gosling, Emma Stone, John Legend",
     "overview": "A jazz pianist and an aspiring actress fall in love while chasing their dreams in Los Angeles.",
     "review": "Dreamy and romantic with a bittersweet, memorable ending."},
    {"title": "The Grand Budapest Hotel", "year": 2014, "genre": "Comedy, Adventure", "rating": 8.1,
     "cast": "Ralph Fiennes, Tony Revolori, F. Murray Abraham",
     "overview": "A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.",
     "review": "Quirky, stylish and laugh-out-loud funny."},
    {"title": "Whiplash", "year": 2014, "genre": "Drama, Music", "rating": 8.5,
     "cast": "Miles Teller, J.K. Simmons, Melissa Benoist",
     "overview": "A young drummer is pushed to his limits by a ruthless music teacher.",
     "review": "Tense and electrifying; you will be gripping your seat."},
    {"title": "Get Out", "year": 2017, "genre": "Horror, Thriller", "rating": 7.7,
     "cast": "Daniel Kaluuya, Allison Williams, Bradley Whitford",
     "overview": "A young man visiting his girlfriend's family slowly uncovers a disturbing secret.",
     "review": "Clever, creepy social thriller that keeps you guessing."},
    {"title": "Coco", "year": 2017, "genre": "Animation, Family", "rating": 8.4,
     "cast": "Anthony Gonzalez, Gael Garcia Bernal, Benjamin Bratt",
     "overview": "A boy who dreams of becoming a musician enters the Land of the Dead to uncover his family's history.",
     "review": "Colorful and full of music; keep tissues nearby."},
    {"title": "The Martian", "year": 2015, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Matt Damon, Jessica Chastain, Chiwetel Ejiofor",
     "overview": "An astronaut stranded on Mars uses science and humor to survive until rescue.",
     "review": "Funny, optimistic and a joy for science lovers."},
    {"title": "Arrival", "year": 2016, "genre": "Sci-Fi, Drama", "rating": 7.9,
     "cast": "Amy Adams, Jeremy Renner, Forest Whitaker",
     "overview": "A linguist works to communicate with alien visitors before fear pushes the world toward war.",
     "review": "Thoughtful and moving; a slow-burn sci-fi that rewards patience."},
    {"title": "Back to the Future", "year": 1985, "genre": "Sci-Fi, Comedy", "rating": 8.5,
     "cast": "Michael J. Fox, Christopher Lloyd, Lea Thompson",
     "overview": "A teenager is sent thirty years into the past in a time-traveling car and must make sure his parents fall in love.",
     "review": "Pure fun: clever, fast and endlessly rewatchable."},
    {"title": "Amelie", "year": 2001, "genre": "Romance, Comedy", "rating": 8.3,
     "cast": "Audrey Tautou, Mathieu Kassovitz, Rufus",
     "overview": "A shy Parisian waitress secretly improves the lives of the people around her while searching for love.",
     "review": "Charming, whimsical and uplifting."},
    {"title": "Gladiator", "year": 2000, "genre": "Action, Drama", "rating": 8.5,
     "cast": "Russell Crowe, Joaquin Phoenix, Connie Nielsen",
     "overview": "A betrayed Roman general becomes a gladiator and seeks revenge on the corrupt emperor.",
     "review": "Grand, gritty and emotional with a powerful lead."},
    {"title": "The Silence of the Lambs", "year": 1991, "genre": "Thriller, Crime", "rating": 8.6,
     "cast": "Jodie Foster, Anthony Hopkins, Scott Glenn",
     "overview": "A young FBI trainee asks an imprisoned psychiatrist for help in catching a serial killer.",
     "review": "Chilling and suspenseful with unforgettable performances."},
    {"title": "Up", "year": 2009, "genre": "Animation, Adventure", "rating": 8.3,
     "cast": "Ed Asner, Jordan Nagai, Christopher Plummer",
     "overview": "An elderly widower ties balloons to his house and flies to South America with an unexpected young stowaway.",
     "review": "The opening alone will make you cry; the rest will make you smile."},
    {"title": "Knives Out", "year": 2019, "genre": "Mystery, Comedy", "rating": 7.9,
     "cast": "Daniel Craig, Chris Evans, Ana de Armas",
     "overview": "A detective investigates the death of a wealthy novelist surrounded by a family of suspects.",
     "review": "A witty, twisty whodunit that is great fun to guess along with."},
    {"title": "Everything Everywhere All at Once", "year": 2022, "genre": "Sci-Fi, Comedy", "rating": 7.8,
     "cast": "Michelle Yeoh, Stephanie Hsu, Ke Huy Quan",
     "overview": "A laundromat owner is pulled across parallel universes to stop a threat that could unravel reality.",
     "review": "Wild, inventive and surprisingly touching."},
    {"title": "The Pursuit of Happyness", "year": 2006, "genre": "Drama", "rating": 8.0,
     "cast": "Will Smith, Jaden Smith, Thandiwe Newton",
     "overview": "A struggling salesman and his young son face homelessness while he works toward a career in finance.",
     "review": "Inspiring and heartfelt, anchored by a warm father-son bond."},
    {"title": "Dune", "year": 2021, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Timothee Chalamet, Rebecca Ferguson, Oscar Isaac",
     "overview": "A noble young heir travels to a dangerous desert planet to protect his family and its people.",
     "review": "Visually stunning and immersive on a big screen."},
    {"title": "Paddington 2", "year": 2017, "genre": "Family, Comedy", "rating": 7.8,
     "cast": "Ben Whishaw, Hugh Bonneville, Sally Hawkins",
     "overview": "A polite bear is wrongly jailed for a theft and his adoptive family works to prove his innocence.",
     "review": "Sweet, funny and kind-hearted; pure comfort viewing."},
    {"title": "Your Name", "year": 2016, "genre": "Animation, Romance", "rating": 8.4,
     "cast": "Ryunosuke Kamiki, Mone Kamishiraishi, Ryo Narita",
     "overview": "Two teenagers mysteriously swap bodies and try to find each other across distance and time.",
     "review": "Gorgeous animation and a romance that hits hard."},
    {"title": "Oppenheimer", "year": 2023, "genre": "Drama, History", "rating": 8.3,
     "cast": "Cillian Murphy, Emily Blunt, Matt Damon",
     "overview": "The story of the physicist who led the creation of the first atomic bomb and of its aftermath.",
     "review": "Dense, intense and gripping; a big, serious drama."},
]

print("Movies in our knowledge base:", len(movies))
movies[0]

## 2. Turn each movie into text

The embedding model reads text, so we describe every movie as one string: title, year, genre, rating, cast, plot and review. Everything we want to be searchable goes in.
The same string is also what the agent reads when the search tool returns a movie.

In [ ]:
def movie_to_text(movie):
    """Describe one movie as a single line of text."""
    return (f"{movie['title']} ({movie['year']}) | Genre: {movie['genre']} | Rating: {movie['rating']} "
            f"| Cast: {movie['cast']} | Plot: {movie['overview']} | Review: {movie['review']}")


movie_texts = []
for movie in movies:
    movie_texts.append(movie_to_text(movie))

print(movie_texts[3])

## 3. Embeddings and the ChromaDB vector store

An **embedding** is a list of numbers that captures the *meaning* of a text. Texts with similar meaning get similar numbers, so "a funny cartoon for kids" ends up close to *Toy Story*, even though they share no words.

**ChromaDB** is a vector database. It stores the vectors together with each movie's details and quickly finds the closest ones.
The first run downloads the embedding model (about 90 MB).

In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
movie_vectors = embedder.encode(movie_texts)

print("Shape:", movie_vectors.shape)                       # (32 movies, 384 numbers each)
print("First 5 numbers of movie 0:", movie_vectors[0][:5])

In [ ]:
def build_collection(movies, vectors, texts):
    """Create an in-memory ChromaDB collection holding our movies and their vectors."""
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="movie_collection")

    collection.add(
        ids=[str(number) for number in range(len(movies))],
        embeddings=vectors.tolist(),        # we computed the vectors ourselves
        documents=texts,
        metadatas=movies,                   # each movie dictionary is stored next to its vector
    )
    return collection


collection = build_collection(movies, movie_vectors, movie_texts)
print("Movies stored in ChromaDB:", collection.count())

## 4. Retrieval function

Embed the question, then ask ChromaDB for the closest movies. A **smaller distance means a closer match**.

In [ ]:
def search_movies_chroma(question, k=TOP_K):
    """Return the k movies whose meaning is closest to the question."""
    question_vector = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=question_vector, n_results=k)

    found = []
    for movie, distance in zip(results["metadatas"][0], results["distances"][0]):
        movie = dict(movie)
        movie["distance"] = round(float(distance), 3)
        found.append(movie)
    return found


def show_movies(found):
    """Print the movies as simple info cards."""
    for movie in found:
        print(f"{movie['title']} ({movie['year']})  |  Rating: {movie['rating']}  |  {movie['genre']}  |  distance {movie['distance']}")
        print(f"   Cast:    {movie['cast']}")
        print(f"   Summary: {movie['overview']}")
        print(f"   Review:  {movie['review']}")
        print()


show_movies(search_movies_chroma("a funny animated movie the whole family can enjoy", 3))

## 5. Turn retrieval into a **tool**

A tool is a normal Python function that the agent is allowed to call. ADK reads three things from it:

1. the **function name**
2. the **docstring**, which tells the agent *when* to use it
3. the **type hints**, which tell the agent what to pass in

So the docstring matters: it is the instruction manual for the tool.

The tool also remembers the movies it found in `last_found`, so we can show posters later.

In [ ]:
last_found = []      # the movies from the most recent search


def search_movies_tool(question: str) -> str:
    """Search the movie database. Use this to find movies by genre, cast, plot, mood or review.

    Args:
        question: What the user is looking for, for example "animated family movies".

    Returns:
        A text list of the best matching movies.
    """
    global last_found
    last_found = search_movies_chroma(question)

    result = ""
    for movie in last_found:
        result = result + "- " + movie_to_text(movie) + "\n"
    return result


# A tool is just a function, so we can test it without any agent.
print(search_movies_tool("space adventure")[:400])

## 6. Create the agent

An ADK `Agent` needs four things:

| Part | What it is |
|---|---|
| `name` | a short identifier |
| `model` | the LLM that does the thinking |
| `instruction` | the agent's job description (this replaces our hand-written prompt) |
| `tools` | the functions it may call |

Notice there is no `build_prompt` any more. The retrieved movies reach the LLM as the tool's result.

In [ ]:
movie_agent = Agent(
    name="movie_assistant",
    model=LLM_MODEL,
    instruction=(
        "You are a friendly movie assistant. "
        "ALWAYS use the search_movies_tool to find movies before answering. "
        "Answer using ONLY the movies the tool returns. "
        "If the user wants recommendations, pick the best 3 and give one short reason for each. "
        "If the tool results do not contain the answer, say so honestly."
    ),
    tools=[search_movies_tool],
)

print("Agent ready:", movie_agent.name)

## 7. Run the agent

A **runner** runs the agent: it sends our question in, lets the agent call tools as needed, and returns a list of *events* (the question, the tool call, the tool result, the final answer).

The runner keeps a **session**, so the agent remembers earlier questions. Try a follow-up like *"which of those is the shortest?"*.

`await` works directly in a notebook cell.

In [ ]:
runner = InMemoryRunner(agent=movie_agent)


async def ask_agent(question):
    """Send a question to the agent and return its final answer as text."""
    events = await runner.run_debug(question, quiet=True)

    answer = ""
    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            answer = event.content.parts[0].text
    return answer

In [ ]:
question = "Something like Interstellar, but shorter and more hopeful"

answer = await ask_agent(question)

print("Question:", question)
print()
print("Movies the tool found:")
show_movies(last_found)
print("Agent's answer:")
print(answer)

**Your turn:** change the question and run it again. Try a genre, an actor, or a mood.

In [ ]:
answer = await ask_agent("Movies with Tom Hanks")

show_movies(last_found)
print(answer)

### Watch the agent think

Print every event to see the agent's steps: it calls the tool, gets the movies back, then writes the answer. This is the part simple RAG did not have.

In [ ]:
events = await runner.run_debug("Recommend a romantic movie", quiet=True)

for event in events:
    for part in event.content.parts:
        if part.function_call:
            print("AGENT CALLS TOOL:", part.function_call.name, part.function_call.args)
        elif part.function_response:
            print("TOOL RETURNED movies")
        elif part.text:
            print(event.author.upper() + ":", part.text[:300])

## Bonus A: show the posters

This part is only for looks, it is not RAG. It looks up each poster on OMDb, then on TMDb, and shows a plain picture if neither has it.
Both services give free API keys (https://www.omdbapi.com/apikey.aspx and https://www.themoviedb.org/settings/api). You can use your own.

In [ ]:
OMDB_KEY = "fc1fef96"
TMDB_KEY = "f5a0494cf81e7d1e1f13b65a953af255"


def read_json(url):
    """Download a web address and return its JSON content."""
    with urllib.request.urlopen(url, timeout=5) as response:
        return json.loads(response.read().decode())


def poster_from_omdb(title, year):
    """Return a poster url from OMDb, or None."""
    url = f"https://www.omdbapi.com/?t={urllib.parse.quote(title)}&y={year}&apikey={OMDB_KEY}"
    try:
        data = read_json(url)
        if data.get("Poster", "N/A").startswith("http"):
            return data["Poster"]
    except Exception as error:
        print("OMDb failed for", title, "->", error)
    return None


def poster_from_tmdb(title, year):
    """Return a poster url from TMDb, or None."""
    url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_KEY}&query={urllib.parse.quote(title)}&year={year}"
    try:
        results = read_json(url).get("results", [])
        if len(results) > 0 and results[0].get("poster_path"):
            return "https://image.tmdb.org/t/p/w300" + results[0]["poster_path"]
    except Exception as error:
        print("TMDb failed for", title, "->", error)
    return None


def get_poster_url(title, year):
    """Try OMDb, then TMDb, then a plain placeholder picture."""
    poster = poster_from_omdb(title, year)
    if poster is None:
        poster = poster_from_tmdb(title, year)
    if poster is None:
        poster = "https://placehold.co/180x260/0f172a/ffffff?text=" + urllib.parse.quote(title)
    return poster


def movies_to_html(found):
    """Build one HTML card per movie, with the poster on the left."""
    cards = ""
    for movie in found:
        poster = get_poster_url(movie["title"], movie["year"])
        cards = cards + f"""
        <div style="display: flex; margin-bottom: 16px; border: 1px solid #ddd; padding: 12px; border-radius: 8px; background: #f9f9f9; color: #222;">
            <img src="{poster}" alt="{movie['title']} poster" style="width: 180px; margin-right: 18px; border-radius: 6px; object-fit: cover;"
                 onerror="this.src='https://placehold.co/180x260?text=No+Image'">
            <div>
                <h3 style="margin: 0; font-size: 1.35em;">{movie['title']} ({movie['year']})</h3>
                <p style="margin: 6px 0;"><b>Rating:</b> ⭐ {movie['rating']} | <b>Genre:</b> {movie['genre']}</p>
                <p style="margin: 6px 0;"><b>Cast:</b> {movie['cast']}</p>
                <p style="margin: 6px 0;">{movie['overview']}</p>
                <p style="margin: 6px 0; font-style: italic; color: #555;">"{movie['review']}"</p>
            </div>
        </div>
        """
    return cards


display(HTML(movies_to_html(search_movies_chroma("a funny animated movie the whole family can enjoy", 3))))

## Bonus B: a tiny web app with Gradio

The same `ask_agent` function behind a text box. Gradio accepts `async` functions, so no extra code is needed.

In [ ]:
import gradio as gr


async def web_search(question):
    """What the web app runs when someone presses Submit."""
    answer = await ask_agent(question)
    return answer, movies_to_html(last_found)


demo = gr.Interface(
    fn=web_search,
    inputs=gr.Textbox(label="What kind of movie do you want?", placeholder="e.g. a heartwarming animated movie for the whole family"),
    outputs=[gr.Markdown(label="Answer"), gr.HTML(label="Movies")],
    title="🎬 Movie RAG Agent (Google ADK)",
)

demo.launch()

## Simple RAG vs ADK agent

| | Simple RAG | ADK agent |
|---|---|---|
| Who decides to search? | our code, always | the agent |
| Who writes the prompt? | `build_prompt` | the agent's `instruction` |
| Search query | the user's exact words | the agent can rephrase it |
| Several searches per question? | no | yes, if the agent needs them |
| Follow-up questions | needs extra code | the session remembers |
| Cost | 1 LLM call | 2 or more LLM calls |

## What to try next

- **Tool docstring:** make it vague (`"Search stuff."`) and see whether the agent still calls it.
- **Ask something that needs no search** (for example *"hello"*). Does the agent skip the tool?
- **Ask for a comparison** (*"compare Toy Story and Up"*). Does it search more than once?
- **Add a second tool**, for example `get_top_rated(genre)`, and watch the agent choose between them.
- **Add a memory check:** ask a follow-up such as *"which of those is the shortest?"* and look at what the session remembers.